# Kim Konuşuyor? — Model A (sıfırdan CNN) vs Model B (pretrained ECAPA)

Bu notebook iki konuşmacı tanıma modelini **aynı veriyle** eğitir ve sahte sohbetlerde **DER** ile karşılaştırır.
Hücreleri yukarıdan aşağı sırayla çalıştır. Her hücrenin üstünde ne yaptığı ve çıktıda neye bakman gerektiği yazıyor.

Veri `hazirla.py` ile hazırlandı (Aşama 2): `data/manifest.csv`, `data/chunks/`, `data/sohbetler/`.
Diarization ve model tanımları `ortak.py` içinde; aynı kod Aşama 5'teki Gradio uygulamasında da kullanılacak.

## 0. Kurulum
Kütüphaneleri yükler, 48 çekirdeği PyTorch'a verir ve rastgeleliği sabitler (aynı sonuçları tekrar alabilmek için).

**Çıktıda bak:** hata olmamalı; PyTorch sürümü `2.6.0+cpu` ve 48 thread yazmalı.

In [ ]:
import base64, copy, io, json, time, warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
from IPython.display import HTML, Audio, display
from torch.utils.data import DataLoader, Dataset

from ortak import (SR, HeadB, LogMel, build_model_a, diarize, ecapa_embed, load_ecapa, read_rttm,
                   to_annotation)

warnings.filterwarnings("ignore")
torch.set_num_threads(48)
torch.manual_seed(0); np.random.seed(0)
ROOT = Path.cwd()
(ROOT / "models").mkdir(exist_ok=True)
print("PyTorch", torch.__version__, "| thread:", torch.get_num_threads())

## 1. Veriyi yükle
`data/manifest.csv` her 2 saniyelik parçanın dosya yolunu, kişisini, oturumunu ve hangi bölüme (train / val / test) ait olduğunu tutar.
Bütün parçalar belleğe alınır (yaklaşık 2.200 parça × 32.000 örnek).

**Çıktıda bak:** her kişinin train, val ve test sütunu dolu olmalı. Train sayısı augmentation'lı kopyaları da içerir (1 temiz + 3 değiştirilmiş).
Test oturumu (3) eğitimde **hiç** kullanılmaz.

In [ ]:
man = pd.read_csv(ROOT / "data" / "manifest.csv")
PEOPLE = sorted(man.person.unique())
P2I = {p: i for i, p in enumerate(PEOPLE)}
man["y"] = man.person.map(P2I)
wavs = np.stack([sf.read(ROOT / p, dtype="float32")[0] for p in man.path])

tr = (man.split == "train").values
va = (man.split == "val").values
te = (man.split == "test").values
print("Kişiler:", PEOPLE, "| toplam parça:", len(man))
display(pd.crosstab(man.person, man.split)[["train", "val", "test"]])

## 2. Her kişiden bir örnek: dinle ve spektrogramına bak
Spektrogram sesi resme çevirir: **yatay eksen zaman**, **dikey eksen frekans** (aşağısı kalın, yukarısı ince sesler), **renk ses şiddeti** (parlak = güçlü).
Model A tam olarak bu resme bakacak.

**Çıktıda bak:** kişilerin parlak yatay çizgileri (sesin perdesi ve harmonikleri) farklı yüksekliklerde mi? Onur'un
spektrogramında üst frekanslarda kara boşluklar var: iPad'in sıkıştırması (Aşama 2'deki not).

In [ ]:
logmel = LogMel()


def spec_png(wav, w=4.2, h=1.6):
    with torch.no_grad():
        s = logmel(torch.from_numpy(wav)[None])[0, 0].numpy()
    fig, ax = plt.subplots(figsize=(w, h))
    ax.imshow(s, origin="lower", aspect="auto", cmap="magma", extent=[0, len(wav) / SR, 0, 64])
    ax.set_xlabel("zaman (sn)"); ax.set_ylabel("mel"); fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=90); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


def audio_b64(wav):
    buf = io.BytesIO(); sf.write(buf, wav, SR, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode()


def yan_yana(rows):
    """rows: [(başlık, ses)] → her satırda başlık | dinleme düğmesi | spektrogram."""
    html = "<table>" + "".join(
        f"<tr><td style='padding-right:12px'><b>{t}</b></td>"
        f"<td><audio controls src='data:audio/wav;base64,{audio_b64(w)}'></audio></td>"
        f"<td><img src='data:image/png;base64,{spec_png(w)}'></td></tr>" for t, w in rows) + "</table>"
    display(HTML(html))


ornek = [man[(man.person == p) & tr & (man.aug == "temiz")].index[20] for p in PEOPLE]
yan_yana([(f"{man.person[i]}<br><small>{Path(man.path[i]).name}</small>", wavs[i]) for i in ornek])

## 3. Augmentation örnekleri
Aşama 2'de her train parçasından 1 temiz + 3 değiştirilmiş kopya diske yazıldı: **gürültü** (gerçek oda sesi), **yankı** (sentetik oda yankısı) ve **seviye** (parça içinde yavaşça değişen ses seviyesi).
İki model de bu kopyalarla eğitilecek.

**Çıktıda bak:** dört versiyonda da aynı kişi aynı şeyi söylüyor olmalı. Gürültülü olanın spektrogramında arka plan daha "karlı", yankılı olanda sesler zamanda bulanık.

In [ ]:
base = man.loc[ornek[0]]
same = man[(man.person == base.person) & (man.session == base.session) & (man.chunk == base.chunk)]
yan_yana([(r.aug, wavs[i]) for i, r in same.iterrows()])

## 4. Model A için veri: spektrogramlar ve `to_rgb`
Her parçanın log-mel spektrogramı bir kez hesaplanır (`[1, 64 mel, 201 zaman]`).
- `Parcalar` dataset'inin `targets` listesi her parçanın kişisini tutar (canlı panel bununla her kişiden örnek seçer).
- `to_rgb` spektrogramı panelde gösterilecek renkli bir resme çevirir: 0–1 aralığında, `(H, W, 3)`, magma paleti.

**Çıktıda bak:** `feats` boyutu `(parça sayısı, 1, 64, 201)` olmalı; altta bir örneğin `to_rgb` hali görünür.

In [ ]:
with torch.no_grad():
    feats = torch.cat([logmel(torch.from_numpy(wavs[i:i + 512])) for i in range(0, len(wavs), 512)])
labels = torch.tensor(man.y.values)
print("feats:", tuple(feats.shape))


class Parcalar(Dataset):
    def __init__(self, mask):
        idx = np.where(mask)[0]
        self.x, self.y = feats[idx], labels[idx]
        self.targets = self.y.tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], int(self.y[i])


CMAP = matplotlib.colormaps["magma"]


def to_rgb(t):
    """(1, mel, zaman) → (H, W, 3) 0..1; düşük frekanslar altta."""
    s = t[0].flip(0)
    s = (s - s.min()) / (s.max() - s.min() + 1e-9)
    return torch.from_numpy(CMAP(s.numpy())[..., :3]).float()


train_ds, val_ds, test_ds = Parcalar(tr), Parcalar(va), Parcalar(te)
print("train / val / test:", len(train_ds), len(val_ds), len(test_ds))
plt.figure(figsize=(5, 2)); plt.imshow(to_rgb(val_ds[0][0])); plt.axis("off"); plt.title(PEOPLE[val_ds[0][1]]); plt.show()

## 5. Model A: sıfırdan CNN
Aşama 3'te onaylanan mimari: 5 blok (**Conv 3×3 → BatchNorm → ReLU → MaxPool**), kanallar 16-32-64-64-128,
sonra **global average pooling** (spektrogramın tamamının ortalaması: model sesin *nerede* olduğunu değil *nasıl* olduğunu öğrenir), Dropout ve tek bir Linear.
Ağırlıklar rastgele başlar.

**Çıktıda bak:** her bloktan sonra resim yarıya küçülür (64×201 → 2×6); parametre sayısı ~135 bin.

In [ ]:
model_a = build_model_a(len(PEOPLE))
h = feats[:1]
for i, layer in enumerate(model_a[:5], 1):
    h = layer(h); print(f"Blok {i}: {tuple(h.shape[1:])}")
print(model_a[5:])
print("Parametre:", sum(p.numel() for p in model_a.parameters()))

## 6. Model A'yı eğit (canlı panel açık)
Eğitim başlayınca canlı panel açılır. Tarayıcıda yeni bir sekmede **https://student.exposureai.org/lab/proxy/8002/** adresini aç
(hücre çıktısı başka bir port yazarsa adresteki 8002'yi onunla değiştir, sondaki `/` kalsın).
Damga spektrogramın üstünden geçer, kuleler yükselir, sonunda kişilerin isim çubukları dolar.

- 30 epoch, AdamW (lr 1e-3), batch 64, `num_workers=4`.
- Her epoch sonunda validation (oturum 4) ölçülür; validation loss'u **en düşük** epoch'un ağırlıkları saklanır. (Accuracy'ye göre seçseydik, doğru bilip emin olmadığı erken bir epoch seçilebilirdi; loss modelin ne kadar emin ve doğru olduğunu birlikte ölçer.)

**Çıktıda bak:** `train_acc` ve `val_acc` birlikte yükselmeli. Train %100'e yaklaşıp val çok düşük kalıyorsa model kayıtları ezberliyor (**overfitting**): bunu görürsen Claude'a sor.

In [ ]:
from live import CNNLive

EPOCHS_A = 30


@torch.no_grad()
def evaluate(model, ds):
    model.eval()
    out = model(ds.x)
    return F.cross_entropy(out, ds.y).item(), (out.argmax(1) == ds.y).float().mean().item()


def train_a(model, epochs, lr, view=None):
    dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, persistent_workers=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * len(dl))
    history, best = [], (float("inf"), None, 0)
    for ep in range(1, epochs + 1):
        model.train()
        tl = tc = n = 0
        for i, (x, y) in enumerate(dl, 1):
            out = model(x)
            loss = F.cross_entropy(out, y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); n += len(y)
            if view: view(model, history, i, len(dl), x=x, y=y)
        vl, vacc = evaluate(model, val_ds)
        history.append(dict(epoch=ep, train_loss=tl / n, train_acc=tc / n, val_loss=vl, val_acc=vacc))
        if vl < best[0]:
            best = (vl, copy.deepcopy(model.state_dict()), ep)
        print(f"epoch {ep:2d}  train loss {tl/n:.3f} acc {tc/n:.3f} | val loss {vl:.3f} acc {vacc:.3f}")
        if view: view(model, history, len(dl), len(dl), force=True)
    model.load_state_dict(best[1])
    print(f"En düşük val loss {best[0]:.3f} (epoch {best[2]}, val acc {history[best[2] - 1]['val_acc']:.3f}) ağırlıkları yüklendi.")
    return history, best[2]


torch.manual_seed(0)
model_a = build_model_a(len(PEOPLE))
view_a = CNNLive(model_a, val_ds, to_rgb, PEOPLE, epochs=EPOCHS_A)
t0 = time.time()
hist_a, best_ep_a = train_a(model_a, EPOCHS_A, 1e-3, view=view_a)
time_a = time.time() - t0
view_a.close()
print(f"Model A eğitim süresi: {time_a:.0f} sn")

## 7. Model A eğitim grafiği
Sol: loss (düşmeli), sağ: accuracy (yükselmeli). Mavi train, turuncu validation. Kesikli çizgi saklanan epoch.

**Çıktıda bak:** iki eğri arasındaki açıklık. Train ile val arasındaki fark büyüdükçe ezberleme artar.

In [ ]:
def plot_history(hist, best_ep, title):
    h = pd.DataFrame(hist)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    for k, a in zip(("loss", "acc"), ax):
        a.plot(h.epoch, h[f"train_{k}"], label="train"); a.plot(h.epoch, h[f"val_{k}"], label="validation")
        a.axvline(best_ep, ls="--", c="gray"); a.set_xlabel("epoch"); a.set_title(f"{title} — {k}"); a.legend()
    fig.tight_layout(); plt.show()


plot_history(hist_a, best_ep_a, "Model A")

## 8. Model B: ECAPA embedding'lerini çıkar
`speechbrain/spkrec-ecapa-voxceleb` 7.000'den fazla kişinin sesiyle önceden eğitilmiş. Her 2 sn parçayı **192 sayılık bir ses parmak izine** (embedding) çevirir.
ECAPA **donuk**: ağırlıkları hiç değişmez. Her parçanın embedding'i bir kez hesaplanıp `data/emb_ecapa.npy` dosyasına kaydedilir; notebook'u tekrar çalıştırınca diskten okunur.

Model B'de canlı panel yok: ECAPA eğitilmediği için izlenecek bir öğrenme yok. Pretraining'in farkı da tam olarak bu.

**Çıktıda bak:** `emb` boyutu `(parça sayısı, 192)`. İlk çalıştırmada yaklaşık 1–2 dakika sürer.

In [ ]:
ecapa = load_ecapa()
cache = ROOT / "data" / "emb_ecapa.npy"
if cache.exists() and np.load(cache, mmap_mode="r").shape[0] == len(man):
    emb = np.load(cache)
    print("Embedding'ler diskten okundu.")
else:
    t0 = time.time()
    emb = ecapa_embed(ecapa, wavs)
    np.save(cache, emb)
    print(f"Embedding çıkarma: {time.time() - t0:.0f} sn")
emb_t = torch.from_numpy(emb)
print("emb:", emb.shape)

## 9. Model B'nin sınıflandırıcısını eğit
Embedding'in üstüne tek bir Linear katman (192 → kişi sayısı). Girdi önce L2-normalize edilir (sadece parmak izinin *yönü* önemli, büyüklüğü değil).
Model A ile **aynı** train parçaları (4 kopya dahil) ve aynı validation kullanılır. Validation loss'u en düşük epoch saklanır.

**Çıktıda bak:** saniyeler içinde biter. Eğitilen parametre sayısını Model A ile karşılaştır.

In [ ]:
def train_b(head, epochs=300, lr=1e-2):
    xtr, ytr = emb_t[tr], labels[tr]
    xva, yva = emb_t[va], labels[va]
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    history, best = [], (float("inf"), None, 0)
    for ep in range(1, epochs + 1):
        head.train()
        out = head(xtr); loss = F.cross_entropy(out, ytr)
        opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            ov = head(xva)
            vl, vacc = F.cross_entropy(ov, yva).item(), (ov.argmax(1) == yva).float().mean().item()
        history.append(dict(epoch=ep, train_loss=loss.item(), train_acc=(out.argmax(1) == ytr).float().mean().item(),
                            val_loss=vl, val_acc=vacc))
        if vl < best[0]:
            best = (vl, copy.deepcopy(head.state_dict()), ep)
    head.load_state_dict(best[1])
    print(f"En düşük val loss {best[0]:.3f} (epoch {best[2]}, val acc {history[best[2] - 1]['val_acc']:.3f})")
    return history, best[2]


torch.manual_seed(0)
head_b = HeadB(len(PEOPLE))
t0 = time.time()
hist_b, best_ep_b = train_b(head_b)
time_b = time.time() - t0
print(f"Model B sınıflandırıcı eğitimi: {time_b:.1f} sn | eğitilen parametre: {sum(p.numel() for p in head_b.parameters())}")
plot_history(hist_b, best_ep_b, "Model B")

## 10. Test parçalarında karışıklık matrisi
Test = oturum 3 (laptop, orta mesafe; demo da böyle yapılacak). Eğitimde hiç görülmedi.
Satır: gerçek kişi, sütun: modelin tahmini. Köşegen doğru bilinenler.

**Çıktıda bak:** köşegen dışındaki kutular modelin **kimi kiminle karıştırdığını** gösterir. Sesleri benzeyen iki kişi varsa en çok onları karıştırır.
Onur satırına da bak: test oturumu da iPad'den, yani burada iPad izi Onur'u kolay tanıtıyor olabilir; asıl kontrol Aşama 5'teki canlı test.

In [ ]:
@torch.no_grad()
def test_preds():
    model_a.eval(); head_b.eval()
    pa = model_a(test_ds.x).softmax(1).numpy()
    pb = head_b(emb_t[te]).softmax(1).numpy()
    return pa, pb


pa, pb = test_preds()
yt = labels[te].numpy()
acc_a, acc_b = (pa.argmax(1) == yt).mean(), (pb.argmax(1) == yt).mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, p, name, acc in zip(axes, (pa, pb), ("Model A — sıfırdan CNN", "Model B — ECAPA"), (acc_a, acc_b)):
    cm = np.zeros((len(PEOPLE), len(PEOPLE)), int)
    for t, q in zip(yt, p.argmax(1)):
        cm[t, q] += 1
    ax.imshow(cm, cmap="Blues")
    for i in range(len(PEOPLE)):
        for j in range(len(PEOPLE)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(len(PEOPLE)), PEOPLE); ax.set_yticks(range(len(PEOPLE)), PEOPLE)
    ax.set_xlabel("tahmin"); ax.set_ylabel("gerçek"); ax.set_title(f"{name}\ntest acc {acc:.1%}")
fig.tight_layout(); plt.show()

## 11. Sahte sohbetlerde DER
Model sohbeti **2 sn pencereyle, 0,5 sn kaydırarak** dinler; her 0,5 sn'nin kişisi, üstüne düşen pencerelerin olasılık ortalamasıyla seçilir (`ortak.diarize`).
Konuşmanın nerede olduğunu iki model için de **aynı** Silero VAD bulur.

DER'i `pyannote.metrics`'in `IdentificationErrorRate`'i hesaplar (isimler olduğu gibi karşılaştırılır). Üç hatanın toplamıdır:
- **Yanlış kişi:** Ayşe konuşurken model "Mert" dedi
- **Kaçırılan konuşma:** biri konuşurken model sessizlik dedi
- **Yanlış alarm:** sessizlikte model birinin konuştuğunu söyledi

**Çıktıda bak:** iki modelin kaçırılan konuşma ve yanlış alarm oranı aynı olmalı (aynı VAD). Aradaki farkı **yanlış kişi** belirler. 0'a yakın = iyi.

In [ ]:
from pyannote.metrics.identification import IdentificationErrorRate
from ortak import speech_regions


@torch.no_grad()
def predict_a(x):
    model_a.eval()
    return model_a(logmel(torch.from_numpy(x))).softmax(1).numpy()


@torch.no_grad()
def predict_b(x):
    head_b.eval()
    return head_b(torch.from_numpy(ecapa_embed(ecapa, x))).softmax(1).numpy()


PREDICTS = {"A": predict_a, "B": predict_b}
conv_paths = sorted((ROOT / "data" / "sohbetler").glob("sohbet_*.wav"))
totals = {k: dict(confusion=0.0, **{"missed detection": 0.0, "false alarm": 0.0}, total=0.0) for k in PREDICTS}
per_conv, diar = [], {}
t0 = time.time()
for p in conv_paths:
    wav = sf.read(p, dtype="float32")[0]
    ref = read_rttm(p.with_suffix(".rttm"))
    regions = speech_regions(wav)
    d = diarize(wav, PREDICTS, PEOPLE, regions)
    diar[p.stem] = d
    row = {"sohbet": p.stem}
    for k in PREDICTS:
        r = IdentificationErrorRate()(ref, to_annotation(d[k], p.stem), detailed=True)
        for c in totals[k]:
            totals[k][c] += r[c]
        row[f"DER {k}"] = r["identification error rate"]
    per_conv.append(row)
print(f"{len(conv_paths)} sohbet, {time.time() - t0:.0f} sn")

der = {}
for k, t in totals.items():
    der[k] = dict(DER=(t["confusion"] + t["missed detection"] + t["false alarm"]) / t["total"],
                  **{"yanlış kişi": t["confusion"] / t["total"], "kaçırılan": t["missed detection"] / t["total"],
                     "yanlış alarm": t["false alarm"] / t["total"]})
display(pd.DataFrame(der).T.rename(index={"A": "Model A — sıfırdan CNN", "B": "Model B — ECAPA"}).style.format("{:.1%}"))
display(pd.DataFrame(per_conv).set_index("sohbet").style.format("{:.1%}"))

## 12. Bir sahte sohbetin zaman çizelgeleri
Üstte doğru cevap (RTTM), altında Model A ve Model B. Her kişi kendi renginde; bölümlerin üstünde isim ve modelin ortalama **% confidence**'ı.
Sohbeti dinleyerek çizelgeyi takip edebilirsin.

**Çıktıda bak:** konuşmacı değiştiği anlarda modeller ne kadar gecikiyor, hangi kişide yanlış renge geçiyor? `SOHBET` değişkenini değiştirerek başka sohbetlere bak.

In [ ]:
SOHBET = "sohbet_01"
COLORS = dict(zip(PEOPLE, ["#7c3aed", "#0d9488", "#f59e0b", "#ef4444", "#2563eb", "#65a30d"]))


def plot_timelines(name):
    ref = read_rttm(ROOT / "data" / "sohbetler" / f"{name}.rttm")
    rows = [("Doğru cevap", [(s.start, s.end, w, None) for s, _, w in ref.itertracks(yield_label=True)]),
            ("Model A — sıfırdan CNN", diar[name]["A"]), ("Model B — ECAPA", diar[name]["B"])]
    fig, ax = plt.subplots(figsize=(16, 3.2))
    for r, (title, segs) in enumerate(rows):
        y = len(rows) - 1 - r
        for a, b, who, conf in segs:
            ax.barh(y, b - a, left=a, height=0.7, color=COLORS[who])
            if b - a >= 1.2:
                txt = who if conf is None else f"{who}\n%{conf * 100:.0f}"
                ax.text(a + 0.1, y, txt, va="center", fontsize=7.5, color="white", fontweight="bold")
    ax.set_yticks(range(len(rows)), [t for t, _ in rows][::-1]); ax.set_xlabel("zaman (sn)"); ax.set_xlim(0, 60)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORS.values()], labels=list(COLORS),
              loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=len(COLORS), frameon=False)
    fig.tight_layout(); plt.show()


plot_timelines(SOHBET)
display(Audio(str(ROOT / "data" / "sohbetler" / f"{SOHBET}.wav")))

## 13. Karşılaştırma tablosu
İki modelin tüm sonuçları yan yana. Pencere süresi = bir 2 sn pencerenin CPU'da işlenme süresi (64 pencerelik bir batch'in ortalaması).

**Çıktıda bak:** DER'i düşük olan önde. Bu tabloyu `DENEY_GUNLUGU.md`'ye ekleyeceğiz.

In [ ]:
def ms_per_window(fn, n=64):
    x = np.stack([wavs[i] for i in np.where(te)[0][:n]])
    fn(x[:4]); t0 = time.time(); fn(x)
    return (time.time() - t0) / n * 1000


tablo = pd.DataFrame({
    "Model A — sıfırdan CNN": {
        "Eğitilen parametre": sum(p.numel() for p in model_a.parameters()),
        "Eğitim süresi (sn)": round(time_a),
        "Val acc (seçilen epoch)": f"{hist_a[best_ep_a - 1]['val_acc']:.1%}",
        "Test ort. güven": f"{pa.max(1).mean():.0%}",
        "Test acc": f"{acc_a:.1%}", "DER": f"{der['A']['DER']:.1%}", "Yanlış kişi": f"{der['A']['yanlış kişi']:.1%}",
        "Pencere süresi (ms)": round(ms_per_window(predict_a), 1)},
    "Model B — ECAPA": {
        "Eğitilen parametre": sum(p.numel() for p in head_b.parameters()),
        "Eğitim süresi (sn)": round(time_b, 1),
        "Val acc (seçilen epoch)": f"{hist_b[best_ep_b - 1]['val_acc']:.1%}",
        "Test ort. güven": f"{pb.max(1).mean():.0%}",
        "Test acc": f"{acc_b:.1%}", "DER": f"{der['B']['DER']:.1%}", "Yanlış kişi": f"{der['B']['yanlış kişi']:.1%}",
        "Pencere süresi (ms)": round(ms_per_window(predict_b), 1)},
})
display(tablo)

## 14. Modelleri kaydet
İki model `models/` klasörüne `.pt` olarak kaydedilir; Aşama 5'teki Gradio uygulaması (`app.py`) buradan yükler.
Model B için sadece sınıflandırıcı kaydedilir: ECAPA'nın kendisi değişmediği için her seferinde aynı yerden indirilir.
Sonuçlar `models/sonuclar.json` dosyasına da yazılır (web uygulamasındaki bilgi kartı için).

**Çıktıda bak:** iki dosyanın yolu ve boyutu yazmalı.

In [ ]:
torch.save(dict(state_dict=model_a.state_dict(), names=PEOPLE, channels=[16, 32, 64, 64, 128],
                history=hist_a, best_epoch=best_ep_a), ROOT / "models" / "model_a.pt")
torch.save(dict(state_dict=head_b.state_dict(), names=PEOPLE, l2_normalize_embedding=True,
                history=hist_b, best_epoch=best_ep_b), ROOT / "models" / "model_b_head.pt")
json.dump({k: {m: round(v, 4) for m, v in d.items()} | {"test_acc": round(float(a), 4)}
           for (k, d), a in zip(der.items(), (acc_a, acc_b))},
          open(ROOT / "models" / "sonuclar.json", "w"), indent=1, ensure_ascii=False)
for f in ("model_a.pt", "model_b_head.pt", "sonuclar.json"):
    print(f"models/{f}: {(ROOT / 'models' / f).stat().st_size / 1024:.0f} KB")